# Цвиль Павел. Фит-231 Лабораторная работа №4

### Задание 1

Возьмите реализацию класса HashTable из лекционных материалов и выполните
следующие доработки:
1. Реализуйте квадратичное пробирование как технику повторного хеширования.
2. Реализуйте работу с функцией len (переопределите метод __len__).
3. Реализуйте работу оператора in (переопределите метод __contains__).
4. Переделайте метод put таким образом, чтобы таблица автоматически меняла размер,
когда загрузочный фактор становится больше значения 0.7. Размер должен
увеличиваться примерно в два раза до ближайшего подходящего простого числа.
5. Реализуйте работу оператора del (переопределите метод __delitem__) для удаления
элемента таблицы. Таблица должна автоматически менять размер, когда
загрузочный фактор становится меньше значения 0.2. Размер должен уменьшаться
примерно в два раза до ближайшего подходящего простого числа.
Все выполненные доработки должны быть протестированы.

In [4]:
class HashTable:
    def __init__(self, size=11):
        self.size = size                    # размер таблицы
        self.slots = [None] * self.size     # ключи
        self.data = [None] * self.size      # значения
        self.count = 0                      # количество элементов
    
    def hashfunction(self, key, size):
        return key % size
    
    def rehash(self, key, size, attempt):
        """Квадратичное пробирование"""
        return (key + attempt**2) % size
    
    def _find_next_prime(self, n):
        """Находит следующее простое число"""
        def is_prime(num):
            if num < 2:
                return False
            for i in range(2, int(num**0.5) + 1):
                if num % i == 0:
                    return False
            return True
        
        while not is_prime(n):
            n += 1
        return n
    
    def _resize(self, new_size):
        """Изменяет размер таблицы"""
        old_slots = self.slots
        old_data = self.data
        old_count = self.count
        
        self.size = self._find_next_prime(new_size)
        self.slots = [None] * self.size
        self.data = [None] * self.size
        self.count = 0
        
        for i in range(len(old_slots)):
            if old_slots[i] is not None:
                self.put(old_slots[i], old_data[i])
    
    def put(self, key, value):
        load = self.count / self.size
        if load >= 0.7:
            self._resize(self.size * 2)
        
        hash_val = self.hashfunction(key, self.size)
        attempt = 0
        
        while self.slots[hash_val] is not None and self.slots[hash_val] != key:
            attempt += 1
            hash_val = self.rehash(key, self.size, attempt)
        
        if self.slots[hash_val] is None:
            self.count += 1
        self.slots[hash_val] = key
        self.data[hash_val] = value
    
    def get(self, key):
        hash_val = self.hashfunction(key, self.size)
        attempt = 0
        start = hash_val
        
        while self.slots[hash_val] is not None:
            if self.slots[hash_val] == key:
                return self.data[hash_val]
            attempt += 1
            hash_val = self.rehash(key, self.size, attempt)
            if hash_val == start:
                break
        return None
    
    def __getitem__(self, key):
        return self.get(key)
    
    def __setitem__(self, key, value):
        self.put(key, value)
    
    def __len__(self):
        return self.count
    
    def __contains__(self, key):
        return self.get(key) is not None
    
    def __delitem__(self, key):
        hash_val = self.hashfunction(key, self.size)
        attempt = 0
        start = hash_val
        
        while self.slots[hash_val] is not None:
            if self.slots[hash_val] == key:
                self.slots[hash_val] = None
                self.data[hash_val] = None
                self.count -= 1
                
                # Уменьшаем размер если загрузка < 0.2
                if self.count > 0 and self.count / self.size < 0.2 and self.size > 11:
                    self._resize(max(11, self.size // 2))
                return
            attempt += 1
            hash_val = self.rehash(key, self.size, attempt)
            if hash_val == start:
                break
        raise KeyError(key)
    
    def __str__(self):
        result = []
        for i in range(self.size):
            if self.slots[i] is not None:
                result.append(f"{self.slots[i]}: {self.data[i]}")
        return "{" + ", ".join(result) + "}"


print("ЗАДАНИЕ 1: Хеш-таблица с квадратичным пробированием")

ht = HashTable(11)
print("1. Добавление элементов:")
for i, val in enumerate([54, 26, 93, 17, 77, 31, 44, 55, 20]):
    ht.put(val, f"value_{val}")
    print(f"   put({val}) -> загрузка: {ht.count/ht.size:.2f}")

print(f"2. Таблица: {ht}")
print(f"   len(ht) = {len(ht)}")

print("3. Проверка через in:")
print(f"   77 in ht = {77 in ht}")
print(f"   100 in ht = {100 in ht}")

print("4. Получение значений:")
print(f"   ht[77] = {ht[77]}")
print(f"   ht[44] = {ht[44]}")

print("5. Удаление элемента:")
del ht[77]
print(f"   После del ht[77]: {ht}")
print(f"   len(ht) = {len(ht)}")
print(f"   77 in ht = {77 in ht}")

print("6. Добавление до автоматического увеличения:")
for i in range(20, 30):
    ht.put(i, f"{i}")
print(f"   Таблица после добавления: {ht}")
print(f"   Размер таблицы: {ht.size}")

ЗАДАНИЕ 1: Хеш-таблица с квадратичным пробированием
1. Добавление элементов:
   put(54) -> загрузка: 0.09
   put(26) -> загрузка: 0.18
   put(93) -> загрузка: 0.27
   put(17) -> загрузка: 0.36
   put(77) -> загрузка: 0.45
   put(31) -> загрузка: 0.55
   put(44) -> загрузка: 0.64
   put(55) -> загрузка: 0.73
   put(20) -> загрузка: 0.39
2. Таблица: {93: value_93, 26: value_26, 77: value_77, 55: value_55, 54: value_54, 31: value_31, 17: value_17, 20: value_20, 44: value_44}
   len(ht) = 9
3. Проверка через in:
   77 in ht = True
   100 in ht = False
4. Получение значений:
   ht[77] = value_77
   ht[44] = value_44
5. Удаление элемента:
   После del ht[77]: {93: value_93, 26: value_26, 55: value_55, 54: value_54, 31: value_31, 17: value_17, 20: value_20, 44: value_44}
   len(ht) = 8
   77 in ht = False
6. Добавление до автоматического увеличения:
   Таблица после добавления: {22: 22, 93: value_93, 24: 24, 26: 26, 23: 23, 27: 27, 25: 25, 29: 29, 55: value_55, 54: value_54, 31: value_31, 28:

### Задание 2 

Возьмите реализацию класса HashTable из лекционных материалов и выполните
следующие доработки:
1. Переделайте существующие методы так, чтобы разрешение коллизий происходило
не при помощи концепции открытой адресации, а методом цепочек. Для этого в
каждом слоте храните связный список, реализованный классом UnorderedList из
лабораторной работы 3.
2. Реализуйте работу с функцией len (переопределите метод __len__).
3. Реализуйте работу оператора in (переопределите метод __contains__).
4. Переделайте метод put таким образом, чтобы таблица автоматически меняла размер,
когда загрузочный фактор становится больше значения 0.7. Размер должен
увеличиваться примерно в два раза до ближайшего подходящего простого числа.
5. Реализуйте работу оператора del (переопределите метод __delitem__) для удаления
элемента таблицы. Таблица должна автоматически менять размер, когда
загрузочный фактор становится меньше значения 0.2. Размер должен уменьшаться
примерно в два раза до ближайшего подходящего простого числа.
Все выполненные доработки должны быть протестированы.



In [1]:
class Node:
    def __init__(self, data):
        self.data = data
        self.next = None

class UnorderedList:
    def __init__(self):
        self.head = None
    
    def isEmpty(self):
        return self.head == None
    
    def add(self, item):
        temp = Node(item)
        temp.next = self.head
        self.head = temp
    
    def __str__(self):
        if self.head == None:
            return "[]"
        result = []
        current = self.head
        while current != None:
            result.append(str(current.data))
            current = current.next
        return "[" + ", ".join(result) + "]"


class HashTableChained:
    def __init__(self, size=11):
        self.size = size
        self.table = [UnorderedList() for _ in range(size)]
        self.count = 0
    
    def _hash(self, key):
        return key % self.size
    
    def put(self, key, value):
        h = self._hash(key)
        bucket = self.table[h]
        
        current = bucket.head
        while current:
            if current.data[0] == key:
                current.data = (key, value)
                return
            current = current.next
        
        bucket.add((key, value))
        self.count += 1
    
    def get(self, key):
        h = self._hash(key)
        bucket = self.table[h]
        
        current = bucket.head
        while current:
            if current.data[0] == key:
                return current.data[1]
            current = current.next
        return None
    
    def __str__(self):
        result = []
        for i, bucket in enumerate(self.table):
            if not bucket.isEmpty():
                items = []
                current = bucket.head
                while current:
                    items.append(f"{current.data[0]}: {current.data[1]}")
                    current = current.next
                result.append(f"слот {i}: " + " -> ".join(items))
        return "\n".join(result)



ht = HashTableChained(7)

while True:
    print("  1 - добавить пару (ключ значение)")
    print("  2 - найти значение по ключу")
    print("  3 - показать таблицу")
    print("  4 - выйти")
    
    choice = input("Ваш выбор: ")
    
    if choice == "1":
        key = int(input("  Введите ключ (целое число): "))
        value = input("  Введите значение: ")
        ht.put(key, value)
        print(f"  Добавлено: {key} -> {value}")
    
    elif choice == "2":
        key = int(input("  Введите ключ: "))
        val = ht.get(key)
        if val:
            print(f"  Значение: {val}")
        else:
            print(f"  Ключ {key} не найден")
    
    elif choice == "3":
        print("\nХеш-таблица:")
        print(ht)
    
    elif choice == "4":
        print("Выход")
        break

  1 - добавить пару (ключ значение)
  2 - найти значение по ключу
  3 - показать таблицу
  4 - выйти
  Добавлено: 52 -> 1
  1 - добавить пару (ключ значение)
  2 - найти значение по ключу
  3 - показать таблицу
  4 - выйти
  Добавлено: 67 -> 2
  1 - добавить пару (ключ значение)
  2 - найти значение по ключу
  3 - показать таблицу
  4 - выйти
  Ключ 3 не найден
  1 - добавить пару (ключ значение)
  2 - найти значение по ключу
  3 - показать таблицу
  4 - выйти
  Добавлено: 2 -> 7
  1 - добавить пару (ключ значение)
  2 - найти значение по ключу
  3 - показать таблицу
  4 - выйти
  Значение: 7
  1 - добавить пару (ключ значение)
  2 - найти значение по ключу
  3 - показать таблицу
  4 - выйти
Выход


### Задание 3

Переделайте класс HashTable, чтобы в качестве ключей можно было использовать строки


In [ ]:
class Node:
    def __init__(self, data):
        self.data = data
        self.next = None

class UnorderedList:
    def __init__(self):
        self.head = None
    
    def isEmpty(self):
        return self.head == None
    
    def add(self, item):
        temp = Node(item)
        temp.next = self.head
        self.head = temp
    
    def __str__(self):
        if self.head == None:
            return "[]"
        result = []
        current = self.head
        while current != None:
            result.append(str(current.data))
            current = current.next
        return "[" + ", ".join(result) + "]"


class HashTableString:
    def __init__(self, size=7):
        self.size = size
        self.table = [UnorderedList() for _ in range(size)]
        self.count = 0
    
    def _hash(self, key):
        total = 0
        for ch in key:
            total += ord(ch)
        return total % self.size
    
    def put(self, key, value):
        h = self._hash(key)
        bucket = self.table[h]
        
        current = bucket.head
        while current:
            if current.data[0] == key:
                current.data = (key, value)
                return
            current = current.next
        
        bucket.add((key, value))
        self.count += 1
    
    def get(self, key):
        h = self._hash(key)
        bucket = self.table[h]
        
        current = bucket.head
        while current:
            if current.data[0] == key:
                return current.data[1]
            current = current.next
        return None
    
    def __str__(self):
        result = []
        for i in range(self.size):
            bucket = self.table[i]
            if not bucket.isEmpty():
                items = []
                current = bucket.head
                while current:
                    items.append(f"{current.data[0]}: {current.data[1]}")
                    current = current.next
                result.append(f"слот {i}: " + " -> ".join(items))
            else:
                result.append(f"слот {i}: пусто")
        return "\n".join(result)


ht = HashTableString(7)

while True:
    print("\n  1 - добавить пару (ключ значение)")
    print("  2 - найти значение по ключу")
    print("  3 - показать таблицу")
    print("  4 - выйти")
    
    choice = input("Ваш выбор: ")
    
    if choice == "1":
        key = input("  Введите ключ (строку): ")
        value = input("  Введите значение: ")
        ht.put(key, value)
        print(f"  Добавлено: {key} -> {value}")
    
    elif choice == "2":
        key = input("  Введите ключ: ")
        val = ht.get(key)
        if val:
            print(f"  Значение: {val}")
        else:
            print(f"  Ключ '{key}' не найден")
    
    elif choice == "3":
        print("\nХеш-таблица:")
        print(ht)
    
    elif choice == "4":
        print("Выход")
        break


  1 - добавить пару (ключ значение)
  2 - найти значение по ключу
  3 - показать таблицу
  4 - выйти
  Добавлено: кот -> 1

  1 - добавить пару (ключ значение)
  2 - найти значение по ключу
  3 - показать таблицу
  4 - выйти
  Добавлено: пёс -> 2

  1 - добавить пару (ключ значение)
  2 - найти значение по ключу
  3 - показать таблицу
  4 - выйти

Хеш-таблица:
слот 0: пусто
слот 1: пусто
слот 2: пусто
слот 3: кот: 1
слот 4: пусто
слот 5: пёс: 2
слот 6: пусто

  1 - добавить пару (ключ значение)
  2 - найти значение по ключу
  3 - показать таблицу
  4 - выйти
Выход


### Задание 4

Дана строчка русского текста, состоящая из слов и пробелов. Словом считается
последовательность русских букв, слова разделены одним или большим числом пробелов.
Для каждого слова этого текста узнайте порядковый номер его вхождения в текст именно в
той форме, в которой указано слово. Для первого вхождения слова выведите «1», для
второго вхождения того же слова выведите «2» и так далее.
Для решения этой задачи используйте класс HashTable из задания № 3

In [6]:
class Node:
    def __init__(self, data):
        self.data = data
        self.next = None

class UnorderedList:
    def __init__(self):
        self.head = None
    
    def isEmpty(self):
        return self.head == None
    
    def add(self, item):
        temp = Node(item)
        temp.next = self.head
        self.head = temp
    
    def search(self, item):
        current = self.head
        while current != None:
            if current.data[0] == item:
                return current.data[1]
            current = current.next
        return None
    
    def update(self, item, value):
        current = self.head
        while current != None:
            if current.data[0] == item:
                current.data = (item, value)
                return True
            current = current.next
        return False
    
    def __str__(self):
        if self.head == None:
            return "[]"
        result = []
        current = self.head
        while current != None:
            result.append(str(current.data))
            current = current.next
        return "[" + ", ".join(result) + "]"


class HashTableString:
    def __init__(self, size=7):
        self.size = size
        self.table = [UnorderedList() for _ in range(size)]
        self.count = 0
    
    def _hash(self, key):
        total = 0
        for ch in key:
            total += ord(ch)
        return total % self.size
    
    def get(self, key):
        h = self._hash(key)
        bucket = self.table[h]
        return bucket.search(key)
    
    def put(self, key, value):
        h = self._hash(key)
        bucket = self.table[h]
        
        # Если ключ уже есть - обновляем
        if bucket.update(key, value):
            return
        
        # Если нет - добавляем
        bucket.add((key, value))
        self.count += 1
    
    def __str__(self):
        result = []
        for i, bucket in enumerate(self.table):
            if not bucket.isEmpty():
                result.append(f"слот {i}: {bucket}")
        return "\n".join(result)



text = input("Введите текст: ")

words = text.split()
ht = HashTableString(11)
result = []

for word in words:
    count = ht.get(word)
    if count is None:
        ht.put(word, 1)
        result.append(1)
    else:
        ht.put(word, count + 1)
        result.append(count + 1)

print("\nРезультат:")
print(" ".join(map(str, result)))


Результат:
1 2 3 1 2 1 2 4 5


### Задание 5

Напишите программу, имитирующую процесс регистрации и авторизации. Для каждого
пользователя программа должна сохранять логин, хеш его пароля и «соль». Для хранения
данных можно использовать БД или файл.
Действия при сохранении пароля:
1. Сгенерируйте длинную случайную «соль» при помощи модуля secrets (secrets —
Generate secure random numbers for managing secrets — Python 3.10.0 documentation).
Длина «соли» должна быть такой же как и выходные данные используемой вами
хэш-функции. Например, если для хеширования вы используете SHA256, то на
выходе вы получите 256 бит (32 байта). В этом случае соль должна составлять не
менее 32 случайных байт.
2. Добавьте «соль» к паролю и хэшируйте его с помощью функции scrypt из модуля
hashlib (Хеширование паролей модулем hashlib в Python. (docs-python.ru)).
3. Сохраните логин, «соль» и получившейся хэш в БД или в файл. 

In [9]:
import secrets
import hashlib
import os

# Файл для хранения данных
DB_FILE = "users.txt"

def hash_password(password, salt=None):
    """Хеширует пароль с солью"""
    if salt is None:
        salt = secrets.token_bytes(32)  # генерируем соль (32 байта)
    
    # Добавляем соль к паролю и хешируем
    password_bytes = password.encode('utf-8')
    hash_obj = hashlib.scrypt(password_bytes, salt=salt, n=16384, r=8, p=1, dklen=32)
    
    return salt, hash_obj

def register():
    """Регистрация нового пользователя"""
    print("\n--- РЕГИСТРАЦИЯ ---")
    login = input("Логин: ")
    
    # Проверяем, существует ли пользователь
    if os.path.exists(DB_FILE):
        with open(DB_FILE, 'r') as f:
            for line in f:
                if line.startswith(login + ":"):
                    print("Ошибка: пользователь уже существует")
                    return
    
    password = input("Пароль: ")
    salt, hash_val = hash_password(password)
    
    # Сохраняем: логин:соль(hex):хеш(hex)
    with open(DB_FILE, 'a') as f:
        f.write(f"{login}:{salt.hex()}:{hash_val.hex()}\n")
    
    print("Регистрация успешна!")

def login():
    """Авторизация пользователя"""
    print("\n--- АВТОРИЗАЦИЯ ---")
    login = input("Логин: ")
    password = input("Пароль: ")
    
    if not os.path.exists(DB_FILE):
        print("Ошибка: нет зарегистрированных пользователей")
        return False
    
    with open(DB_FILE, 'r') as f:
        for line in f:
            parts = line.strip().split(':')
            if len(parts) == 3 and parts[0] == login:
                saved_login, salt_hex, hash_hex = parts
                salt = bytes.fromhex(salt_hex)
                saved_hash = bytes.fromhex(hash_hex)
                
                # Хешируем введённый пароль с той же солью
                _, test_hash = hash_password(password, salt)
                
                if test_hash == saved_hash:
                    print("Вход выполнен!")
                    return True
                else:
                    print("Неверный пароль")
                    return False
    
    print("Пользователь не найден")
    return False


while True:
    print("\n1 - Регистрация")
    print("2 - Вход")
    print("3 - Выход")
    
    choice = input("Выберите действие: ")
    
    if choice == "1":
        register()
    elif choice == "2":
        login()
    elif choice == "3":
        print("До свидания!")
        break
    else:
        print("Неверный выбор")


1 - Регистрация
2 - Вход
3 - Выход

--- РЕГИСТРАЦИЯ ---
Регистрация успешна!

1 - Регистрация
2 - Вход
3 - Выход

--- АВТОРИЗАЦИЯ ---
Вход выполнен!

1 - Регистрация
2 - Вход
3 - Выход
До свидания!


### Задание 6

Напишите программу, которая принимает от пользователя путь до директории. Для всех
файлов из данной директории должен быть вычислен хеш. Программа должна выявить и
вывести на экран все дубликаты в этой директории (т.е. файлы, у которых одинаковый хеш).

In [8]:
import os
import hashlib

def get_file_hash(filepath):
    """Вычисляет хеш файла (SHA256)"""
    hash_obj = hashlib.sha256()
    with open(filepath, 'rb') as f:
        # Читаем файл кусками по 4096 байт
        for chunk in iter(lambda: f.read(4096), b''):
            hash_obj.update(chunk)
    return hash_obj.hexdigest()

def find_duplicates(directory):
    """Находит дубликаты файлов в директории"""
    hash_map = {}
    
    # Обходим все файлы в директории
    for root, dirs, files in os.walk(directory):
        for filename in files:
            filepath = os.path.join(root, filename)
            try:
                file_hash = get_file_hash(filepath)
                
                if file_hash not in hash_map:
                    hash_map[file_hash] = []
                hash_map[file_hash].append(filepath)
            except Exception as e:
                print(f"Ошибка при чтении {filepath}: {e}")
    
    # Выводим дубликаты
    duplicates_found = False
    for file_hash, paths in hash_map.items():
        if len(paths) > 1:
            duplicates_found = True
            print(f"\nХеш: {file_hash[:16]}...")
            for path in paths:
                print(f"  - {path}")
    
    if not duplicates_found:
        print("Дубликатов не найдено")


path = input("Введите путь к папке: ")

if os.path.exists(path) and os.path.isdir(path):
    find_duplicates(path)
else:
    print("Ошибка: папка не найдена")

Дубликатов не найдено
